# Pipeline de Ingestão da Tabela de Votação

Notebook responsável pela ingestão da tabela **votacao** utilizando PySpark e Delta Lake. O fluxo realiza a leitura dos arquivos CSV brutos, consolida os dados em um DataFrame e grava a camada Bronze no formato Delta.

#### 1. Importação das Bibliotecas

Nesta etapa importamos as bibliotecas necessárias para realizar a ingestão da tabela **consulta_candidatos**. Utilizamos o Spark para processamento distribuído e o Delta Lake para persistência dos dados na camada Bronze.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip

print('Bibliotecas importadas com sucesso!')

Bibliotecas importadas com sucesso!


#### 2. Inicialização da Sessão Spark

Inicializamos a sessão Spark configurada para trabalhar com o Delta Lake. Essa sessão será utilizada durante todo o processo de ingestão da tabela de votação.

In [3]:
builder = (
    SparkSession.builder
        # .master('local[*]')
        .appName('TSE-Analytics-Validation-Analise-Votacao')
        .config(
            'spark.sql.extensions',
            'io.delta.sql.DeltaSparkSessionExtension'
        )
        .config(
            'spark.sql.catalog.spark_catalog',
            'org.apache.spark.sql.delta.catalog.DeltaCatalog'
        )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print('Sessão Spark criada com sucesso com suporte a Delta Lake!')
print(f'Versão do PySpark: {spark.version}')

:: loading settings :: url = jar:file:/usr/local/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/vscode/.ivy2.5.2/cache
The jars for the packages stored in: /home/vscode/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0b291bbf-90df-4148-a686-7900651da2cc;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.0 in central
	found io.delta#delta-storage;4.3.0 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	found org.apache.logging.log4j#log4j-slf4j2-impl;2.25.3 in central
	found org.apache.logging.log4j#log4j-api;2.25.3 in central
	found com.google.code.findbugs#jsr305;3.0.2 in central
	found io.unitycatalog#unitycatalog-hadoop;0.5.0 in central
	found org.apache.logging.log4j#log4j-core;2.25.3 in central
	found io.delta#delta-kernel-api;4.3.0 in

Sessão Spark criada com sucesso com suporte a Delta Lake!
Versão do PySpark: 4.1.1


#### 5. Leitura da Tabela Bronze e Criação da View Temporária

Nesta etapa, carregamos a tabela consulta_candidatos armazenada em formato Delta na camada Bronze para um DataFrame do PySpark. Em seguida, registramos esse DataFrame como uma view temporária chamada candidatos, permitindo a execução de consultas SQL durante as análises e validações do processo de ingestão.

In [4]:
votacao = (spark.read
                .format('delta')
                .load('data/bronze/votacao'))

votacao.createOrReplaceTempView('votacao')

26/07/04 14:04:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [35]:
spark.sql(""" 

    WITH tb_votos_candidatos_municipio AS(
    
        SELECT
            ANO_ELEICAO,
            NM_TIPO_ELEICAO,
            NR_TURNO,
            DS_ELEICAO,
            COALESCE(SG_UF, SG_UE) AS UF,
            NM_MUNICIPIO,
            DS_CARGO,
            SQ_CANDIDATO,
            NM_CANDIDATO,
            NM_URNA_CANDIDATO,
            SG_PARTIDO,
            NM_PARTIDO,
            DS_SIT_TOT_TURNO,
            SUM(QT_VOTOS_NOMINAIS) AS QT_VOTOS_NOMINAIS_TOTAL
        FROM votacao

        GROUP BY ALL
    ),

    tb_votos_candidatos AS (

        SELECT
            ANO_ELEICAO,
            NM_TIPO_ELEICAO,
            NR_TURNO,
            DS_ELEICAO,
            DS_CARGO,
            SQ_CANDIDATO,
            NM_CANDIDATO,
            NM_URNA_CANDIDATO,
            SG_PARTIDO,
            NM_PARTIDO,
            SUM(QT_VOTOS_NOMINAIS) AS QT_VOTOS_NOMINAIS_TOTAL
        FROM votacao

        GROUP BY ALL
    ),

    tb_votacao_cargo_municipio AS (

        SELECT
            ANO_ELEICAO,
            NM_TIPO_ELEICAO,
            NR_TURNO,
            DS_ELEICAO,
            DS_CARGO,
            COALESCE(SG_UF, SG_UE) AS UF,
            NM_MUNICIPIO,
            SUM(QT_VOTOS_NOMINAIS) AS QT_VOTOS_NOMINAIS_TOTAL
        FROM votacao

        GROUP BY ALL

    ),

    tb_proporcao_votos AS (

        SELECT 
            t1.*, 
            t2.QT_VOTOS_NOMINAIS_TOTAL AS QT_VOTOS_NOMINAIS_TOTAL_CANDIDATO,
            t3.QT_VOTOS_NOMINAIS_TOTAL AS QT_VOTOS_NOMINAIS_TOTAL_CARGO_MUNICIPIO,
            ROUND(t1.QT_VOTOS_NOMINAIS_TOTAL / QT_VOTOS_NOMINAIS_TOTAL_CANDIDATO * 100.0, 2) AS PCT_VOTOS_CANDIDATO_TOTAL,
            ROUND(t1.QT_VOTOS_NOMINAIS_TOTAL / QT_VOTOS_NOMINAIS_TOTAL_CARGO_MUNICIPIO * 100.0, 2) AS PCT_VOTOS_CANDIDATO_MUNICIPIO
        FROM tb_votos_candidatos_municipio AS t1

        LEFT JOIN tb_votos_candidatos AS t2
            ON t1.ANO_ELEICAO = t2.ANO_ELEICAO
            AND t1.NM_TIPO_ELEICAO = t2.NM_TIPO_ELEICAO
            AND t1.NR_TURNO = t2.NR_TURNO
            AND t1.DS_ELEICAO = t2.DS_ELEICAO
            AND t1.DS_CARGO = t2.DS_CARGO
            AND t1.SQ_CANDIDATO = t2.SQ_CANDIDATO

        LEFT JOIN tb_votacao_cargo_municipio AS t3
            ON t1.ANO_ELEICAO = t3.ANO_ELEICAO
            AND t1.NM_TIPO_ELEICAO = t3.NM_TIPO_ELEICAO
            AND t1.NR_TURNO = t3.NR_TURNO
            AND t1.DS_ELEICAO = t3.DS_ELEICAO
            AND t1.DS_CARGO = t3.DS_CARGO
            AND t1.UF = t3.UF
            AND t1.NM_MUNICIPIO = t3.NM_MUNICIPIO

        ORDER BY PCT_VOTOS_CANDIDATO_TOTAL DESC

    )

    SELECT * FROM tb_proporcao_votos

    """).show(10)


26/07/04 15:36:25 ERROR Executor: Exception in task 4.0 in stage 294.0 (TID 3250)
org.apache.spark.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file file:/workspaces/tse-analytics-26/data/bronze/votacao/part-00000-1e9d4fbf-5cef-4733-891e-2516e8bac3db-c000.snappy.parquet.  SQLSTATE: KD001
	at org.apache.spark.sql.errors.QueryExecutionErrors$.cannotReadFilesError(QueryExecutionErrors.scala:911)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:142)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at org.apache.spark.sql.execution.FileSourceScanExec$$anon$1.hasNext(DataSourceScanExec.scala:773)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.columnartorow_nextBatch_0$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.hashAgg_doAggregateWithKeys_0$(U

ConnectionRefusedError: [Errno 111] Connection refused

ConnectionRefusedError: [Errno 111] Connection refused